# FANUC gripper-threshold probe (B6, issue #13)

Runs scripts/probe_gripper_threshold.py against the pinned Isaac Lab 3.0 OvPhysX stack to MEASURE where the gripper actually pinches: step-by-step gripper joints + finger contact forces in a scripted teacher pinch (3 seeds) and an empty close, then a threshold sweep against the measured contact and the pre-registered verdict of protocol fanuc-gripper-threshold-b6 (SPEC). Measurement only: no baseline gate, no PPO training.


In [ ]:
from __future__ import annotations
import json, os, platform, re, subprocess, sys, time
from pathlib import Path

working_dir = Path('/kaggle/working')
working_dir.mkdir(exist_ok=True)
report_path = working_dir / 'probe-notebook-report.json'
report = {'commands': []}
def save_report():
    report_path.write_text(json.dumps(report, indent=2))

def run(command, **kwargs):
    started = time.perf_counter()
    result = subprocess.run(command, text=True, capture_output=True, check=False, **kwargs)
    elapsed = round(time.perf_counter() - started, 3)
    report.setdefault('commands', []).append({'argv': command, 'exit_code': result.returncode,
        'duration_seconds': elapsed,
        'stdout_tail': result.stdout[-6000:], 'stderr_tail': result.stderr[-6000:]})
    print(f'[PERF] command_seconds={elapsed} exit={result.returncode} command={command[0]}')
    save_report()
    return result

gpu = run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'])
driver = run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'])
report.update({'python': sys.version, 'platform': platform.platform(),
          'gpus': [line.strip() for line in gpu.stdout.splitlines() if line.strip()],
          'driver_versions': sorted(set(line.strip() for line in driver.stdout.splitlines() if line.strip()))})
if gpu.returncode != 0 or not report['gpus']:
    raise RuntimeError('Kaggle session has no visible NVIDIA GPU; select GPU accelerator.')
caps = []
try:
    import torch
    report['torch'] = torch.__version__
    for device in range(torch.cuda.device_count()):
        caps.append(torch.cuda.get_device_capability(device))
    report['compute_capabilities'] = [list(cap) for cap in caps]
except Exception as exc:
    report['torch_probe_error'] = repr(exc)
driver_text = report['driver_versions'][0] if report['driver_versions'] else '0'
match = re.match(r'(\d+)\.(\d+)(?:\.(\d+))?', driver_text)
driver_tuple = tuple(map(int, match.groups(default='0'))) if match else (0, 0, 0)
report['isaaclab_linux_driver_compatible'] = driver_tuple >= (580, 65, 6)
report['isaaclab_gpu_arch_compatible'] = bool(caps) and all(cap >= (7, 5) for cap in caps)
print(json.dumps(report, indent=2))
save_report()


In [ ]:
import base64, io, zipfile
from pathlib import Path
PROJECT_ZIP_B64 = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PROJECT_ZIP_B64))) as archive:
    archive.extractall('/kaggle/working/fanuc-cell')
print(sorted(p.name for p in Path('/kaggle/working/fanuc-cell/scripts').glob('*.py')))


In [ ]:
import shutil
from pathlib import Path
if not (report['isaaclab_gpu_arch_compatible'] and report['isaaclab_linux_driver_compatible']):
    raise RuntimeError('GPU/driver below the Isaac Lab 3.0 minimum: ' + str(report))
uv_probe = run(['uv', '--version'])
if uv_probe.returncode != 0:
    install_uv = run([sys.executable, '-m', 'pip', 'install', 'uv'])
    if install_uv.returncode != 0:
        raise RuntimeError(install_uv.stderr[-4000:])
source_dir = Path('/kaggle/working/IsaacLab')
clone = run(['git', 'clone', '--depth', '1', '--branch', 'v3.0.0',
             'https://github.com/isaac-sim/IsaacLab.git', str(source_dir)])
if clone.returncode != 0:
    raise RuntimeError('Could not clone Isaac Lab v3.0.0: ' + clone.stderr[-4000:])
sync = run(['uv', 'sync'], cwd=source_dir)
if sync.returncode != 0:
    raise RuntimeError('Isaac Lab dependency installation failed: ' + sync.stderr[-6000:])
project_dir = Path('/kaggle/working/fanuc_cell_lab')
generate = run(['uv', 'run', 'isaaclab', '--new', '--non_interactive',
               '--task_type', 'external', '--project_path', '/kaggle/working',
               '--name', 'fanuc_cell_lab', '--author', 'Pascal Coimbra',
               '--task_name', 'conveyor_pick_place', '--robot_name', 'm10id12',
               '--workflow', 'manager-based:single-agent',
               '--rl_library', 'rsl_rl', '--rl_algorithm', 'ppo'], cwd=source_dir)
if generate.returncode != 0 or not project_dir.is_dir():
    raise RuntimeError('Isaac Lab project generation failed: ' + generate.stderr[-6000:])
overlay = Path('/kaggle/working/fanuc-cell/isaaclab_overlay/src/fanuc_cell_lab')
project_package = project_dir / 'src' / 'fanuc_cell_lab'
shutil.copytree(overlay, project_package, dirs_exist_ok=True)
shutil.copytree('/kaggle/working/fanuc-cell/src/fanuc_cell',
                project_dir / 'src' / 'fanuc_cell', dirs_exist_ok=True)
shutil.copy2('/kaggle/working/fanuc-cell/scripts/probe_gripper_threshold.py',
             project_dir / 'scripts' / 'probe_gripper_threshold.py')
asset_dir = project_package / 'assets' / 'data'
asset_dir.mkdir(parents=True, exist_ok=True)
official_converter = (source_dir / 'scripts/tools/convert_urdf.py').read_text()
converted_source, replacements = re.subn(
    r'(args_cli\.physics\s*=\s*\"isaacsim_physx\"\s+if\s+args_cli\.require_kit\s+else\s+)\"[^\"]+\"',
    r'\1"ovphysx"', official_converter, count=1)
if replacements != 1:
    raise RuntimeError('Pinned Isaac Lab URDF converter changed its backend selection; inspect before conversion.')
converter_script = source_dir / 'scripts/tools/convert_urdf_ovphysx.py'
converter_script.write_text(converted_source)
contact_code = (
    'import json,sys; from pxr import Usd,UsdPhysics,Sdf; '
    'stage=Usd.Stage.Open(sys.argv[1]); '
    'bodies=[p for p in stage.Traverse() if p.HasAPI(UsdPhysics.RigidBodyAPI)]; '
    "[(p.AddAppliedSchema('PhysxRigidBodyAPI'), p.AddAppliedSchema('PhysxContactReportAPI'), "
    " p.CreateAttribute('physxRigidBody:sleepThreshold', Sdf.ValueTypeNames.Float, custom=False).Set(0.0), "
    " p.CreateAttribute('physxContactReport:threshold', Sdf.ValueTypeNames.Float, custom=False).Set(0.0)) "
    " for p in bodies]; "
    "stage.GetRootLayer().Save(); print(json.dumps({'contact_report_bodies':len(bodies)}))"
)
inspect_code = (
    'import json,sys; from pxr import Usd,UsdPhysics; '
    'stage=Usd.Stage.Open(sys.argv[1]); ps=list(stage.Traverse()); '
    'r=sum(p.IsA(UsdPhysics.RevoluteJoint) for p in ps); '
    'j=sum(p.IsA(UsdPhysics.PrismaticJoint) for p in ps); '
    'b=[str(p.GetPath()) for p in ps if p.HasAPI(UsdPhysics.RigidBodyAPI)]; '
    'print(json.dumps({\"revolute_joints\":int(r),\"prismatic_joints\":int(j),\"rigid_bodies\":b}))'
)
def convert(extra_args, out_dir, label):
    conversion = run(['uv', 'run', '--extra', 'importers', '--extra', 'ovphysx', 'isaaclab', '-p',
                     str(converter_script),
                     '/kaggle/working/fanuc-cell/assets/m10id12_robot.urdf',
                     str(out_dir), '--joint-stiffness', '0.0',
                     '--joint-damping', '0.0', '--joint-target-type', 'none', *extra_args], cwd=source_dir)
    if conversion.returncode != 0:
        raise RuntimeError(f'URDF conversion ({label}) failed: ' + conversion.stderr[-6000:])
    candidates = sorted(out_dir.rglob('m10id12_robot.usd*'))
    if not candidates:
        raise RuntimeError(f'URDF conversion ({label}) produced no USD under ' + str(out_dir))
    usd = candidates[0]
    patch = run(['uv', 'run', 'python', '-c', contact_code, str(usd)], cwd=source_dir)
    if patch.returncode != 0:
        raise RuntimeError(f'Contact patch ({label}) failed: ' + patch.stderr[-4000:])
    inspect = run(['uv', 'run', 'python', '-c', inspect_code, str(usd)], cwd=source_dir)
    if inspect.returncode != 0:
        raise RuntimeError(f'USD inspect ({label}) failed: ' + inspect.stderr[-4000:])
    summary = json.loads(inspect.stdout.strip().splitlines()[-1])
    report[f'usd_{label}'] = {'path': str(usd), 'joints': {k: summary[k] for k in ('revolute_joints', 'prismatic_joints')}}
    save_report()
    return usd
usd = convert([], asset_dir, 'canonical')
expected_asset = asset_dir / 'm10id12_robot' / 'm10id12_robot.usd'
if usd.resolve() != expected_asset.resolve():
    expected_asset.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(usd, expected_asset)
report['usd_path'] = str(expected_asset)
save_report()
print(json.dumps({'usd_canonical': report['usd_canonical']}, indent=2))


In [ ]:
import subprocess
from pathlib import Path
probe_env = os.environ.copy()
probe_env['PYTHONPATH'] = str(project_dir / 'src') + os.pathsep + probe_env.get('PYTHONPATH', '')
command = ['uv', 'run', '--extra', 'ovphysx', 'python',
           str(project_dir / 'scripts' / 'probe_gripper_threshold.py'),
           '--device', 'cuda:0', '--seed', '23', '--repetitions', '3',
           '--output', '/kaggle/working/gripper-threshold-report.json']
try:
    result = subprocess.run(command, text=True, capture_output=True, timeout=1800, cwd=source_dir, env=probe_env)
    lines = [line for line in result.stdout.splitlines() if line.startswith('VERDICT_JSON:')]
    if lines:
        report['verdict'] = json.loads(lines[-1][len('VERDICT_JSON:'):])
    else:
        report['verdict'] = {'stage': 'subprocess', 'error': 'no verdict line',
                             'stdout_tail': result.stdout[-3000:], 'stderr_tail': result.stderr[-3000:],
                             'exit_code': result.returncode}
except subprocess.TimeoutExpired:
    report['verdict'] = {'stage': 'subprocess', 'error': 'timeout 1800s'}
save_report()
verdict = report['verdict']
print(f"stage={verdict.get('stage')} verdict={verdict.get('verdict')} error={str(verdict.get('error'))[:200]}")
print('recommendation:', json.dumps(verdict.get('recommendation'), indent=2))
print('pinch_min_joint_p05_m:', verdict.get('pinch_min_joint_p05_m'))
print('contact_loss_max_joint_p05_m:', verdict.get('contact_loss_max_joint_p05_m'))
print('empty_close_stall_min_joint_m:', verdict.get('empty_close_stall_min_joint_m'))
print('band_occupancy_all_steps:', verdict.get('band_occupancy_all_steps'))


In [ ]:
from IPython.display import FileLink, display
display(FileLink('/kaggle/working/gripper-threshold-report.json'))
display(FileLink('/kaggle/working/gripper-threshold-samples.json'))
print('verdict:', report.get('verdict', {}).get('verdict'))
